# 🧰 Bioinformatik-Werkzeuge

**Aufbau dieses Notebooks**
- **A. Alle Beispiele auswerten:** sechs fertige Beispielordner, ein gemeinsamer Bericht mit Tabellen und Diagrammen. Hier anfangen.
- **B. Eigene Daten auswerten:** deine FASTA-Dateien aus Google Drive oder Colab, gleicher Bericht.
- **C. Schritt für Schritt: Sichelzellanämie:** von der DNA über die mRNA zum Protein, Mutationen selbst einbauen.
- **D. Sequenzen von NCBI holen:** Daten per Accession-Nummer herunterladen.
- **E. Einzelwerkzeuge:** die Bausteine einzeln zum Ausprobieren und Nachvollziehen.

**Bedienung:** In eine graue Code-Zelle klicken und **Shift + Enter** drücken. Einstellungen stehen immer oben in der Zelle.
Jede Zelle in A bis D lädt die Werkzeuge selbst neu aus GitHub, du kannst also mit jeder davon anfangen.
Speichern: *Datei → Kopie in GitHub speichern* (oder *Kopie in Drive speichern*).

**Falls doch einmal eine Fehlermeldung wie** `module 'werkzeuge' has no attribute ...` **kommt:** *Laufzeit → Sitzung neu starten* und die Zelle noch einmal ausführen.

---
## A. 📊 Alle Beispiele auswerten: ein gemeinsamer Bericht

Im Repo liegt der Ordner `beispiele/` mit sechs Projekten. Jedes Projekt ist ein eigener Unterordner mit FASTA-Dateien, einer kurzen Beschreibung (`LIESMICH.md`) und passenden Einstellungen (`einstellungen.json`).

| Ordner | Inhalt | Was man sieht |
|---|---|---|
| `01_sichelzellanaemie` | echte HBB-mRNA + 4 bekannte Mutationen (künstlich eingebaut) | Missense, Nonsense, Frameshift; DNA → mRNA → Protein |
| `02_haemoglobin_arten` | HBB von 7 Säugetieren (NCBI RefSeq) | stumme vs. Missense-Mutationen, Stammbaum der Arten |
| `03_coronaviren` | SARS-CoV-2, SARS-CoV-1, MERS (je ca. 30.000 Basen) | große Genome, viele ORFs; wird beim ersten Lauf von NCBI geladen |
| `04_frauenschuh_orchideen` | ITS-Region von 12 Orchideen | Stammbaum nach Gattungen, kein Protein-Gen |
| `05_pest_plasmid` | Plasmid pPCP1 aus *Yersinia pestis* | ORF-Suche in Bakterien (Startcodons ATG/GTG/TTG) |
| `06_chloroplast_arabidopsis` | komplettes Chloroplasten-Genom (154.478 Basen) | Statistik und ORF-Suche für große Sequenzen |

Jedes Projekt durchläuft automatisch: **1** Übersicht (Länge, GC-Gehalt) → **2** ORF-Suche → **3** paarweise Alignments → **4** Variantenanalyse (Mutationen gegenüber einer Referenz, Transkription, Translation, Wirkung aufs Protein) → **5** Stammbaum (UPGMA und Neighbor-Joining).

Am Ende erscheint der **gemeinsame Bericht** direkt unter der Zelle. Er liegt außerdem als `ergebnisse_beispiele/bericht.html` im Dateibereich links (herunterladen und im Browser öffnen). Dauer: etwa 20 Sekunden plus Download der Coronaviren beim ersten Mal.

In [ ]:
# --- Werkzeuge laden: holt immer den neuesten Stand aus GitHub und lädt die Module frisch ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
else:   # vorhandene Kopie exakt auf den GitHub-Stand bringen
    subprocess.run(["git", "-C", "Bioinfo", "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "Bioinfo", "reset", "-q", "--hard", "origin/main"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
for modul in ("werkzeuge", "analyse", "bericht"):
    sys.modules.pop(modul, None)          # alte Version aus dem Speicher werfen
import werkzeuge, analyse, bericht
stand = subprocess.run(["git", "-C", "Bioinfo", "log", "-1", "--format=%h vom %cd", "--date=format:%d.%m.%Y %H:%M"],
                       capture_output=True, text=True).stdout.strip()
print(f"Werkzeuge geladen (Stand {stand})")
# ---------------------------------------------------------------------------------------------

# ============================================================
# EINSTELLUNGEN
# ============================================================
BERICHT_ANZEIGEN = True          # Bericht direkt hier unter der Zelle anzeigen
ZIP_HERUNTERLADEN = False        # True = alle Ergebnisse als ZIP-Datei herunterladen
# ============================================================

ergebnisse, bericht_pfad = analyse.analysiere_alle("Bioinfo/beispiele", "ergebnisse_beispiele")

if BERICHT_ANZEIGEN:
    from IPython.display import HTML, display
    display(HTML(open(bericht_pfad, encoding="utf-8").read()))
if ZIP_HERUNTERLADEN:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive("ergebnisse_beispiele", "zip", "ergebnisse_beispiele"))

---
## B. 📂 Eigene Daten auswerten

Genau wie A, nur mit deinen eigenen Dateien. **So legst du die Daten ab:**

```
Bioinformatik/daten/            ← dieser Ordner wird ausgewertet
├── mein_erstes_projekt/        ← jeder Unterordner = ein Projekt im Bericht
│   ├── sequenzen.fasta         ← beliebig viele FASTA-Dateien (.fasta .fa .fna .fas .txt)
│   ├── einstellungen.json      ← optional, z. B. {"REFERENZ": "...", "AUSSENGRUPPE": ["..."]}
│   ├── accessions.txt          ← optional: NCBI-Nummern, eine pro Zeile, werden automatisch geladen
│   └── LIESMICH.md             ← optional: Beschreibung, erscheint im Bericht
└── zweites_projekt/
    └── ...
```
FASTA-Dateien direkt in `daten/` (ohne Unterordner) werden als ein eigenes Projekt ausgewertet.

**Woher kommen die Daten?** Stell oben in der Zelle `QUELLE` ein:
- `"drive"`: **empfohlen.** Ordner in deinem Google Drive (Standard: `Meine Ablage/Bioinformatik/daten`). Beim ersten Mal fragt Colab nach Zugriff. Die Ergebnisse landen in `Bioinformatik/daten_ergebnisse` und bleiben gespeichert.
- `"colab"`: Ordner `daten` direkt in Colab (über das Ordnersymbol links hochladen). Wird gelöscht, wenn Colab beendet wird.

**Große Sequenzen:** Statistik und ORF-Suche schaffen ganze Bakteriengenome. Alignments laufen bis 20.000 bp pro Sequenz (z. B. komplette mtDNA mit 16.569 bp). Für längere Sequenzen nimmt der Stammbaum automatisch eine alignment-freie k-mer-Distanz.

**Ergebnisse pro Projekt:** `bericht.txt`, `uebersicht.csv`, `orfs.csv`, `orf_proteine.fasta`, `alignments.txt`, `varianten.csv`, `proteine_varianten.fasta`, `mrna_varianten.fasta`, `distanzmatrix.csv`, `baum_*.nwk`, `stammbaum.png`. CSV-Dateien öffnet Excel (Trennzeichen Semikolon).

In [ ]:
# --- Werkzeuge laden: holt immer den neuesten Stand aus GitHub und lädt die Module frisch ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
else:   # vorhandene Kopie exakt auf den GitHub-Stand bringen
    subprocess.run(["git", "-C", "Bioinfo", "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "Bioinfo", "reset", "-q", "--hard", "origin/main"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
for modul in ("werkzeuge", "analyse", "bericht"):
    sys.modules.pop(modul, None)          # alte Version aus dem Speicher werfen
import werkzeuge, analyse, bericht
stand = subprocess.run(["git", "-C", "Bioinfo", "log", "-1", "--format=%h vom %cd", "--date=format:%d.%m.%Y %H:%M"],
                       capture_output=True, text=True).stdout.strip()
print(f"Werkzeuge geladen (Stand {stand})")
# ---------------------------------------------------------------------------------------------

# ============================================================
# EINSTELLUNGEN
# ============================================================
QUELLE = "drive"                 # "drive" oder "colab"
DRIVE_ORDNER = "/content/drive/MyDrive/Bioinformatik/daten"
BERICHT_ANZEIGEN = True
ZIP_HERUNTERLADEN = False

# Gilt für alle Projekte. None = nicht setzen (dann gilt einstellungen.json im Projektordner oder der Standard).
EINSTELLUNGEN = dict(
    ORF_MIN_AMINOSAEUREN = None,     # Standard 100
    STARTCODONS = None,              # Standard ("ATG",); Bakterien: ("ATG", "GTG", "TTG")
    REFERENZ = None,                 # Variantenanalyse: Name der Referenz (Standard: erste Sequenz)
    CDS = None,                      # Gen in der Referenz als (start, ende); Standard: automatisch
    AUSSENGRUPPE = None,             # z. B. ["Cypripedium_calceolus", "Cypripedium_acaule"]
    MIN_IDENTITAET_VARIANTEN = None, # Standard 80 (%)
)
# ============================================================

if QUELLE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ordner = DRIVE_ORDNER
    ausgabe = DRIVE_ORDNER.rstrip("/") + "_ergebnisse"
else:
    ordner, ausgabe = "daten", "ergebnisse"

os.makedirs(ordner, exist_ok=True)
if not any(not f.startswith(".") for f in os.listdir(ordner)):
    print(f"Der Ordner {ordner} ist noch leer. Leg dort FASTA-Dateien oder Projekt-Unterordner ab "
          f"und starte die Zelle erneut.")
else:
    ergebnisse, bericht_pfad = analyse.analysiere_alle(ordner, ausgabe, **EINSTELLUNGEN)
    if BERICHT_ANZEIGEN:
        from IPython.display import HTML, display
        display(HTML(open(bericht_pfad, encoding="utf-8").read()))
    if ZIP_HERUNTERLADEN:
        import shutil
        from google.colab import files
        files.download(shutil.make_archive("ergebnisse", "zip", ausgabe))

---
## C. 🩸 Schritt für Schritt: Sichelzellanämie und β-Thalassämie

**Biologie:** Hämoglobin transportiert Sauerstoff in den roten Blutkörperchen. Das Gen ***HBB*** codiert die β-Kette.
- **Sichelzellanämie (HbS):** Eine einzige Base ist verändert, `c.20A>T`. Dadurch wird das Codon GAG (Glutamat) zu GTG (Valin). Das hydrophobe Valin lässt Hämoglobin ohne Sauerstoff verklumpen, die Blutkörperchen werden sichelförmig.
- **β-Thalassämie (Codon 39):** `c.118C>T` macht aus CAG (Glutamin) das Stoppcodon TAG. Die β-Kette bricht ab, es entsteht kaum funktionsfähiges Hämoglobin.

**Was die Zelle macht:**
1. Die echte menschliche *HBB*-mRNA (NCBI `NM_000518.5`) aus dem Beispielordner lesen.
2. Das Gen finden und **Transkription und Translation** Schritt für Schritt zeigen.
3. Mutationen einbauen, die du in `MUTATIONEN` einträgst. *Die Varianten sind also künstlich hergestellt, entsprechen aber exakt den bekannten Mutationen.*
4. Alles durch die Analyse schicken, die die Wirkung jeder Mutation bestimmt.

**Achtung, Nummerierung:** In Lehrbüchern heißt die Sichelzellmutation oft „Glu6Val“. Die Analyse zeigt `p.Glu7Val`. Beides ist richtig. Die moderne Schreibweise (HGVS) zählt das Start-Methionin mit, das im fertigen Protein später abgespalten wird.

**Zum Ausprobieren:** Trag eine eigene Mutation ein, z. B. `("Test", 3, "G", "A")` macht aus dem Startcodon ATG ein ATA.

In [ ]:
# --- Werkzeuge laden: holt immer den neuesten Stand aus GitHub und lädt die Module frisch ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
else:   # vorhandene Kopie exakt auf den GitHub-Stand bringen
    subprocess.run(["git", "-C", "Bioinfo", "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "Bioinfo", "reset", "-q", "--hard", "origin/main"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
for modul in ("werkzeuge", "analyse", "bericht"):
    sys.modules.pop(modul, None)          # alte Version aus dem Speicher werfen
import werkzeuge, analyse, bericht
stand = subprocess.run(["git", "-C", "Bioinfo", "log", "-1", "--format=%h vom %cd", "--date=format:%d.%m.%Y %H:%M"],
                       capture_output=True, text=True).stdout.strip()
print(f"Werkzeuge geladen (Stand {stand})")
# ---------------------------------------------------------------------------------------------

# ============================================================
# EINSTELLUNGEN: Mutationen als (Name, c.-Position, erwartete Base, neue Base)
# c.1 = das A des Startcodons ATG
# ============================================================
MUTATIONEN = [
    ("HbS_Sichelzellanaemie", 20, "A", "T"),
    ("beta-Thalassaemie_Codon39", 118, "C", "T"),
]
# ============================================================

# 1. Echte HBB-mRNA lesen
datei = "Bioinfo/beispiele/01_sichelzellanaemie/1_HbA_normal_NM_000518.5.fasta"
referenz_name, hba = list(werkzeuge.lese_fasta(datei).items())[0]
print(f"{referenz_name}: {len(hba)} Basen")

# 2. Gen finden, Transkription und Translation
cds_start, cds_ende = werkzeuge.finde_cds(hba)
gen = hba[cds_start - 1:cds_ende]
mrna = werkzeuge.transkribiere(gen)
protein = werkzeuge.uebersetze(gen)
print(f"Gen (CDS): Position {cds_start}-{cds_ende}, {len(gen)} Basen = {len(gen) // 3} Codons")
print("DNA      5'-" + gen[:45] + "...")
print("mRNA     5'-" + mrna[:45] + "...")
print("Codons      " + "".join(mrna[i:i + 3] + " " for i in range(0, 45, 3)) + "...")
print("Protein     " + "".join(f" {a}  " for a in protein[:15]) + "...")
print(f"Protein ({len(protein)} Aminosäuren): {protein}")

# 3. Mutationen einbauen
def mutiere(seq, c_position, erwartet, neu):
    i = cds_start - 1 + c_position - 1          # Python-Index
    if seq[i] != erwartet:
        raise ValueError(f"An c.{c_position} steht {seq[i]}, nicht {erwartet}.")
    return seq[:i] + neu + seq[i + 1:]

ordner = "beispiel_sichelzelle"
os.makedirs(ordner, exist_ok=True)
for f in os.listdir(ordner):
    os.remove(os.path.join(ordner, f))
with open(f"{ordner}/1_referenz.fasta", "w") as f:
    f.write(f">{referenz_name}\n{hba}\n")
with open(f"{ordner}/2_varianten.fasta", "w") as f:
    for name, pos, erwartet, neu in MUTATIONEN:
        f.write(f">{name} (künstlich: c.{pos}{erwartet}>{neu})\n{mutiere(hba, pos, erwartet, neu)}\n")

# 4. Analyse (Text und Bericht)
print()
R = analyse.analysiere_ordner(ordner, "ergebnisse_sichelzelle", REFERENZ=referenz_name, ZEIGE_GRAFIK=False)

---
## D. 🌐 Sequenzen von NCBI holen
Trag Accession-Nummern ein (zu finden auf ncbi.nlm.nih.gov/nuccore). Die Dateien landen in einem Projektordner, den B auswerten kann.
Für Google Drive: `ORDNER = "/content/drive/MyDrive/Bioinformatik/daten/mein_projekt"` (vorher B einmal mit `QUELLE = "drive"` ausführen).
Alternativ: Nummern in eine Datei `accessions.txt` im Projektordner schreiben, dann lädt die Analyse sie selbst.

In [ ]:
# --- Werkzeuge laden: holt immer den neuesten Stand aus GitHub und lädt die Module frisch ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
else:   # vorhandene Kopie exakt auf den GitHub-Stand bringen
    subprocess.run(["git", "-C", "Bioinfo", "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "Bioinfo", "reset", "-q", "--hard", "origin/main"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
for modul in ("werkzeuge", "analyse", "bericht"):
    sys.modules.pop(modul, None)          # alte Version aus dem Speicher werfen
import werkzeuge, analyse, bericht
stand = subprocess.run(["git", "-C", "Bioinfo", "log", "-1", "--format=%h vom %cd", "--date=format:%d.%m.%Y %H:%M"],
                       capture_output=True, text=True).stdout.strip()
print(f"Werkzeuge geladen (Stand {stand})")
# ---------------------------------------------------------------------------------------------

ACCESSIONS = ["NC_045512.2", "NC_004718.3", "NC_019843.3"]   # SARS-CoV-2, SARS-CoV-1, MERS-CoV
ORDNER = "daten/coronaviren"

for acc, fehler in werkzeuge.lade_von_ncbi(ACCESSIONS, ORDNER):
    print(f"{acc}: {'geladen' if fehler is None else 'Fehler: ' + str(fehler)}")

---
# E. 🔧 Einzelwerkzeuge
Die Bausteine, aus denen die Ordner-Analyse besteht, zum Ausprobieren mit eigenen kurzen Sequenzen. Jede Zelle funktioniert für sich allein.

## E1. Sequenz-Umwandler: DNA ↔ RNA, Reverse Complement, Protein
Ändere `eingabe`.

In [ ]:
# 🧬 Sequenz-Umwandler: DNA ↔ RNA und Protein, alles in einem Schritt
# Trag unten bei "eingabe" deine Sequenz ein (DNA oder RNA) und führe die Zelle aus.

eingabe = "ATGGCCATGGCGCCCAGAACTGAGATCAATAGTACCCGTATTAACGGGTGA"

basen = "UCAG"
aminosaeuren = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_tabelle = {}
nummer = 0
for b1 in basen:
    for b2 in basen:
        for b3 in basen:
            codon_tabelle[b1 + b2 + b3] = aminosaeuren[nummer]
            nummer = nummer + 1


def saeubern(sequenz):
    # Leerzeichen und Zeilenumbrüche entfernen, alles in Großbuchstaben
    return "".join(sequenz.split()).upper()


def dna_zu_rna(dna):
    return dna.replace("T", "U")


def rna_zu_dna(rna):
    return rna.replace("U", "T")


def reverse_complement(dna):
    komplement = {"A": "T", "T": "A", "G": "C", "C": "G", "N": "N"}
    ergebnis = ""
    for base in dna:
        ergebnis = komplement[base] + ergebnis
    return ergebnis


def uebersetze(rna):
    protein = ""
    for i in range(0, len(rna) - 2, 3):
        aminosaeure = codon_tabelle.get(rna[i:i+3], "X")   # X = unbekanntes Codon (z. B. mit N)
        if aminosaeure == "*":
            break
        protein = protein + aminosaeure
    return protein


def umwandeln(sequenz):
    sequenz = saeubern(sequenz)
    if "U" in sequenz and "T" in sequenz:
        print("⚠️ Die Sequenz enthält T und U. Das ist weder DNA noch RNA, bitte prüfen.")
        return
    erlaubt = set("ACGTUN")
    falsch = set(sequenz) - erlaubt
    if falsch:
        print("⚠️ Unerlaubte Zeichen:", "".join(sorted(falsch)))
        return

    if "U" in sequenz:
        print("Erkannt: RNA")
        rna = sequenz
        dna = rna_zu_dna(rna)
    else:
        print("Erkannt: DNA (als codierender Strang gelesen)")
        dna = sequenz
        rna = dna_zu_rna(dna)

    protein = uebersetze(rna)
    gc = (dna.count("G") + dna.count("C")) / len(dna) * 100

    print(f"Länge:               {len(dna)} Basen")
    print(f"GC-Gehalt:           {gc:.1f} %")
    print(f"DNA (codierend):     {dna}")
    print(f"RNA (mRNA):          {rna}")
    print(f"Reverse Complement:  {reverse_complement(dna)}")
    print(f"Protein:             {protein}  ({len(protein)} Aminosäuren)")
    if len(dna) % 3 != 0:
        print(f"Hinweis: Die Länge ist nicht durch 3 teilbar, die letzten {len(dna) % 3} Basen werden nicht übersetzt.")
    if not rna.startswith("AUG"):
        print("Hinweis: Die Sequenz beginnt nicht mit dem Startcodon AUG. Übersetzt wurde trotzdem ab der ersten Base.")


umwandeln(eingabe)

## E2. Zwei Stränge vergleichen: Komplementarität und Mutationen
Ändere `strang_1` und `strang_2`.

In [ ]:
# 🧬 Zwei DNA-Stränge vergleichen
# Trag unten zwei Sequenzen ein (DNA oder RNA, je 5'→3') und führe die Zelle aus.
#
# Der Code beantwortet zwei Fragen:
#  1. Passen die beiden Stränge als Doppelstrang zusammen (sind sie komplementär)?
#  2. Wo unterscheiden sie sich (z. B. Mutationen) und was bewirkt das im Protein?

strang_1 = "ATGGCCATTGTAATGGGCCGCTGA"
strang_2 = "ATGGCTATTGTAATGAGCCGCTGA"

basen = "UCAG"
aminosaeuren = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_tabelle = {}
nummer = 0
for b1 in basen:
    for b2 in basen:
        for b3 in basen:
            codon_tabelle[b1 + b2 + b3] = aminosaeuren[nummer]
            nummer = nummer + 1


def saeubern(sequenz):
    # Leerzeichen entfernen, Großbuchstaben, RNA wird als DNA behandelt (U → T)
    return "".join(sequenz.split()).upper().replace("U", "T")


def reverse_complement(dna):
    komplement = {"A": "T", "T": "A", "G": "C", "C": "G", "N": "N"}
    ergebnis = ""
    for base in dna:
        ergebnis = komplement[base] + ergebnis
    return ergebnis


def uebersetze(dna):
    protein = ""
    for i in range(0, len(dna) - 2, 3):
        aminosaeure = codon_tabelle.get(dna[i:i+3].replace("T", "U"), "X")
        protein = protein + aminosaeure      # Stopp wird hier als * mit angezeigt
    return protein


def mutationstyp(base_1, base_2):
    # Transition: Purin ↔ Purin (A↔G) oder Pyrimidin ↔ Pyrimidin (C↔T)
    # Transversion: Purin ↔ Pyrimidin
    purine = "AG"
    if (base_1 in purine) == (base_2 in purine):
        return "Transition"
    return "Transversion"


def vergleiche(s1, s2):
    s1 = saeubern(s1)
    s2 = saeubern(s2)
    print(f"Strang 1: {len(s1)} Basen")
    print(f"Strang 2: {len(s2)} Basen")

    # --- Frage 1: Doppelstrang? ---
    print("\n=== 1. Sind die Stränge komplementär (bilden sie einen Doppelstrang)? ===")
    if s2 == reverse_complement(s1):
        print("✅ Ja. Strang 2 ist das Reverse Complement von Strang 1.")
        print("   5'-" + s1 + "-3'")
        print("   3'-" + s2[::-1] + "-5'")
        return
    print("Nein, die beiden sind kein perfekter Doppelstrang. Ich vergleiche sie deshalb als zwei Versionen derselben Sequenz.")

    # --- Frage 2: Unterschiede ---
    print("\n=== 2. Unterschiede Position für Position ===")
    if len(s1) != len(s2):
        print("⚠️ Die Stränge sind unterschiedlich lang. Ein Vergleich Base für Base ist dann nicht sinnvoll,")
        print("   weil eine Insertion oder Deletion alles danach verschiebt. Dafür braucht man ein Alignment")
        print("   (Needleman-Wunsch), das ist unser nächstes Thema.")
        return

    markierung = ""
    unterschiede = []
    for i in range(len(s1)):
        if s1[i] == s2[i]:
            markierung = markierung + "|"
        else:
            markierung = markierung + " "
            unterschiede.append(i)

    print("Strang 1: " + s1)
    print("          " + markierung)
    print("Strang 2: " + s2)

    identitaet = (len(s1) - len(unterschiede)) / len(s1) * 100
    print(f"\nUnterschiede (Hamming-Distanz): {len(unterschiede)}")
    print(f"Identität: {identitaet:.1f} %")

    if not unterschiede:
        print("Die beiden Sequenzen sind identisch.")
        return

    print("\nPosition  Änderung  Typ           Codon       Aminosäure   Wirkung")
    for i in unterschiede:
        codon_start = (i // 3) * 3                 # Beginn des Codons, in dem die Mutation liegt
        codon_1 = s1[codon_start:codon_start + 3]
        codon_2 = s2[codon_start:codon_start + 3]
        if len(codon_1) < 3:
            print(f"{i + 1:>8}  {s1[i]}→{s2[i]}       {mutationstyp(s1[i], s2[i]):<12}  (kein vollständiges Codon)")
            continue
        as_1 = codon_tabelle.get(codon_1.replace("T", "U"), "X")
        as_2 = codon_tabelle.get(codon_2.replace("T", "U"), "X")
        if as_1 == as_2:
            wirkung = "stumm (gleiche Aminosäure)"
        elif as_2 == "*":
            wirkung = "Nonsense (neues Stoppcodon!)"
        elif as_1 == "*":
            wirkung = "Stoppcodon verloren"
        else:
            wirkung = "Missense (andere Aminosäure)"
        print(f"{i + 1:>8}  {s1[i]}→{s2[i]}       {mutationstyp(s1[i], s2[i]):<12}  {codon_1}→{codon_2}     {as_1}→{as_2}          {wirkung}")

    print("\nProtein 1: " + uebersetze(s1))
    print("Protein 2: " + uebersetze(s2))
    print("(Positionen zählen ab 1, wie in der Biologie. Gelesen wird ab der ersten Base im Leseraster 1.)")


vergleiche(strang_1, strang_2)

## E3. Alignment: Needleman-Wunsch (global) und Smith-Waterman (lokal)
Ändere `sequenz_1`, `sequenz_2` und `MODUS` ("global" oder "lokal").
Hier steht die einfache **lineare** Lückenstrafe, gut zum Nachvollziehen. Die Ordner-Analyse (A/B) nutzt eine **affine** Lückenstrafe (Öffnen teuer, Verlängern billig, wie BLAST), weil sonst viele verstreute 1-Basen-Lücken entstehen, die im Gen wie falsche Frameshifts aussehen.

In [ ]:
# 🧬 Paarweises Sequenz-Alignment: Needleman-Wunsch (global) und Smith-Waterman (lokal)
#
# So benutzt du es: Den ganzen Code in eine Colab-Zelle kopieren, unten bei
# "MEINE EINGABE" zwei Sequenzen eintragen und die Zelle ausführen.
#
# Bewertung (Scoring), lineare Lückenstrafe:
#   gleiche Zeichen (Match)        → +MATCH
#   verschiedene Zeichen (Mismatch)→ +MISMATCH (negativ)
#   Lücke (Gap, Insertion/Deletion)→ +GAP      (negativ, pro Lückenposition)
#
# Funktioniert für DNA, RNA und Proteine (Proteine aber nur mit einfacher
# Match/Mismatch-Bewertung, nicht mit BLOSUM-Matrix).


# ============================================================
# MEINE EINGABE
# ============================================================
sequenz_1 = "ATGGCCATTGTAATGGGCCGCTGA"
sequenz_2 = "ATGGCCATTGTATGGGCCGCTGA"    # gleiche Sequenz, aber ein A fehlt (Deletion)

MATCH = 1
MISMATCH = -1
GAP = -2

MODUS = "global"      # "global" (Needleman-Wunsch) oder "lokal" (Smith-Waterman)
# ============================================================


def saeubern(sequenz):
    return "".join(sequenz.split()).upper()


def bewerte(a, b, match, mismatch):
    if a == b:
        return match
    return mismatch


def leere_matrix(zeilen, spalten):
    # Eine Tabelle (Liste von Listen) voller Nullen
    matrix = []
    for i in range(zeilen):
        matrix.append([0] * spalten)
    return matrix


def needleman_wunsch(s1, s2, match=MATCH, mismatch=MISMATCH, gap=GAP):
    """Globales Alignment: beide Sequenzen werden vollständig von Anfang bis Ende aligniert."""
    n = len(s1)
    m = len(s2)
    F = leere_matrix(n + 1, m + 1)   # F[i][j] = bester Score für s1[:i] gegen s2[:j]

    # 1. Erste Spalte und erste Zeile: nur Lücken
    for i in range(1, n + 1):
        F[i][0] = i * gap
    for j in range(1, m + 1):
        F[0][j] = j * gap

    # 2. Matrix füllen: jede Zelle kommt von links, oben oder schräg oben-links
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diagonal = F[i - 1][j - 1] + bewerte(s1[i - 1], s2[j - 1], match, mismatch)
            oben = F[i - 1][j] + gap        # Lücke in s2
            links = F[i][j - 1] + gap       # Lücke in s1
            F[i][j] = max(diagonal, oben, links)

    # 3. Traceback: von unten rechts zurück nach oben links
    a1 = ""
    a2 = ""
    i = n
    j = m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and F[i][j] == F[i - 1][j - 1] + bewerte(s1[i - 1], s2[j - 1], match, mismatch):
            a1 = s1[i - 1] + a1
            a2 = s2[j - 1] + a2
            i = i - 1
            j = j - 1
        elif i > 0 and F[i][j] == F[i - 1][j] + gap:
            a1 = s1[i - 1] + a1
            a2 = "-" + a2
            i = i - 1
        else:
            a1 = "-" + a1
            a2 = s2[j - 1] + a2
            j = j - 1

    return {"score": F[n][m], "a1": a1, "a2": a2, "matrix": F, "start1": 1, "start2": 1}


def smith_waterman(s1, s2, match=MATCH, mismatch=MISMATCH, gap=GAP):
    """Lokales Alignment: findet den am besten passenden Abschnitt beider Sequenzen."""
    n = len(s1)
    m = len(s2)
    H = leere_matrix(n + 1, m + 1)   # erste Zeile/Spalte bleiben 0

    bester_score = 0
    beste_i = 0
    beste_j = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diagonal = H[i - 1][j - 1] + bewerte(s1[i - 1], s2[j - 1], match, mismatch)
            oben = H[i - 1][j] + gap
            links = H[i][j - 1] + gap
            H[i][j] = max(0, diagonal, oben, links)   # Unterschied zu NW: nie unter 0
            if H[i][j] > bester_score:
                bester_score = H[i][j]
                beste_i = i
                beste_j = j

    # Traceback: vom besten Feld zurück, bis eine 0 erreicht ist
    a1 = ""
    a2 = ""
    i = beste_i
    j = beste_j
    while i > 0 and j > 0 and H[i][j] > 0:
        if H[i][j] == H[i - 1][j - 1] + bewerte(s1[i - 1], s2[j - 1], match, mismatch):
            a1 = s1[i - 1] + a1
            a2 = s2[j - 1] + a2
            i = i - 1
            j = j - 1
        elif H[i][j] == H[i - 1][j] + gap:
            a1 = s1[i - 1] + a1
            a2 = "-" + a2
            i = i - 1
        else:
            a1 = "-" + a1
            a2 = s2[j - 1] + a2
            j = j - 1

    # i, j zeigen jetzt auf die Stelle direkt vor dem Alignment → +1 für biologische Zählung ab 1
    return {"score": bester_score, "a1": a1, "a2": a2, "matrix": H, "start1": i + 1, "start2": j + 1}


def zeige_matrix(s1, s2, matrix):
    """Druckt die Score-Matrix (nur sinnvoll für kurze Sequenzen)."""
    print("       " + "".join(f"{b:>4}" for b in "-" + s2))
    for i in range(len(matrix)):
        zeilen_base = "-" if i == 0 else s1[i - 1]
        print(f"{zeilen_base:>4}   " + "".join(f"{wert:>4}" for wert in matrix[i]))


def zeige_alignment(ergebnis, breite=60):
    a1 = ergebnis["a1"]
    a2 = ergebnis["a2"]
    mitte = ""
    for x, y in zip(a1, a2):
        if x == y:
            mitte = mitte + "|"      # Match
        elif x == "-" or y == "-":
            mitte = mitte + " "      # Lücke
        else:
            mitte = mitte + "."      # Mismatch

    pos1 = ergebnis["start1"]
    pos2 = ergebnis["start2"]
    for k in range(0, len(a1), breite):
        teil1 = a1[k:k + breite]
        teil2 = a2[k:k + breite]
        print(f"Seq1 {pos1:>6}  {teil1}")
        print(f"            {mitte[k:k + breite]}")
        print(f"Seq2 {pos2:>6}  {teil2}")
        print()
        pos1 = pos1 + len(teil1) - teil1.count("-")
        pos2 = pos2 + len(teil2) - teil2.count("-")

    laenge = len(a1)
    matches = mitte.count("|")
    mismatches = mitte.count(".")
    luecken = mitte.count(" ")
    print(f"Score:       {ergebnis['score']}")
    if laenge > 0:
        print(f"Länge:       {laenge}")
        print(f"Identität:   {matches}/{laenge} ({matches / laenge * 100:.1f} %)")
        print(f"Mismatches:  {mismatches}")
        print(f"Lücken:      {luecken}")


def aligniere(s1, s2, modus="global", match=MATCH, mismatch=MISMATCH, gap=GAP):
    s1 = saeubern(s1)
    s2 = saeubern(s2)
    if len(s1) == 0 or len(s2) == 0:
        print("⚠️ Eine der Sequenzen ist leer.")
        return None
    if len(s1) * len(s2) > 25_000_000:
        print("⚠️ Achtung: sehr lange Sequenzen, das kann in reinem Python mehrere Minuten dauern.")

    if modus == "global":
        print("Globales Alignment (Needleman-Wunsch)")
        ergebnis = needleman_wunsch(s1, s2, match, mismatch, gap)
    elif modus == "lokal":
        print("Lokales Alignment (Smith-Waterman)")
        ergebnis = smith_waterman(s1, s2, match, mismatch, gap)
    else:
        print('⚠️ MODUS muss "global" oder "lokal" sein.')
        return None
    print(f"Bewertung: Match {match:+}, Mismatch {mismatch:+}, Lücke {gap:+}\n")

    if len(s1) <= 15 and len(s2) <= 15:
        print("Score-Matrix:")
        zeige_matrix(s1, s2, ergebnis["matrix"])
        print()
    zeige_alignment(ergebnis)
    return ergebnis


ergebnis = aligniere(sequenz_1, sequenz_2, MODUS)

## E4. ORF-Finder: Gene in einer DNA finden
Standardmäßig wird das Plasmid pPCP1 untersucht. Für eigene DNA: `EIGENE_SEQUENZ` ausfüllen.

In [ ]:
# 🧬 ORF-Finder: mögliche Gene in einer DNA-Sequenz finden
#
# Ein ORF (Open Reading Frame, offenes Leseraster) ist ein Abschnitt, der mit einem
# Startcodon beginnt und ohne Unterbrechung bis zu einem Stoppcodon läuft.
# Ein Gen kann auf jedem der beiden Stränge liegen und in jedem von 3 Leserastern
# beginnen → 6 Leseraster werden durchsucht.
#
# So benutzt du es: Ganzen Code in eine Colab-Zelle kopieren und ausführen.
# Standardmäßig wird das Plasmid pPCP1 geladen und die gefundenen ORFs werden mit
# den 10 bei NCBI annotierten Genen verglichen.

import os
import urllib.request

# ============================================================
# EINSTELLUNGEN
# ============================================================
MIN_AMINOSAEUREN = 60            # kürzere ORFs werden ignoriert (zufällige ORFs sind meist kurz)
STARTCODONS = ["ATG"]            # für Bakterien realistischer: ["ATG", "GTG", "TTG"]
STOPPCODONS = ["TAA", "TAG", "TGA"]

# Eigene Sequenz statt pPCP1? Dann hier eintragen (sonst leer lassen):
EIGENE_SEQUENZ = ""
# ============================================================

BASIS_URL = "https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/"
DATEIEN = {"pPCP1_plasmid.fasta": "NC_005816.fna", "pPCP1_gene.fasta": "NC_005816.ffn"}

basen = "TCAG"
aminosaeuren = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_tabelle = {}
nummer = 0
for b1 in basen:
    for b2 in basen:
        for b3 in basen:
            codon_tabelle[b1 + b2 + b3] = aminosaeuren[nummer]   # hier direkt mit DNA-Codons (T statt U)
            nummer = nummer + 1


def lese_fasta(dateiname):
    sequenzen = {}
    name = None
    with open(dateiname) as f:
        for zeile in f:
            zeile = zeile.strip()
            if zeile == "":
                continue
            if zeile.startswith(">"):
                name = zeile[1:]
                sequenzen[name] = ""
            else:
                sequenzen[name] = sequenzen[name] + zeile.upper()
    return sequenzen


def reverse_complement(dna):
    komplement = {"A": "T", "T": "A", "G": "C", "C": "G", "N": "N"}
    ergebnis = ""
    for base in dna:
        ergebnis = komplement.get(base, "N") + ergebnis
    return ergebnis


def uebersetze(dna):
    protein = ""
    for i in range(0, len(dna) - 2, 3):
        protein = protein + codon_tabelle.get(dna[i:i + 3], "X")
    return protein


def finde_orfs_in_strang(seq, startcodons, stoppcodons, min_as):
    """Sucht ORFs in den 3 Leserastern eines Strangs.
    Gibt Liste mit (rahmen, start, ende) zurück; start/ende als Python-Index (ab 0, ende exklusiv).
    Pro Stoppcodon wird nur der längste ORF genommen (erstes Startcodon)."""
    orfs = []
    for rahmen in range(3):
        start = None
        for i in range(rahmen, len(seq) - 2, 3):
            codon = seq[i:i + 3]
            if start is None and codon in startcodons:
                start = i
            elif start is not None and codon in stoppcodons:
                ende = i + 3                         # Stoppcodon gehört zum ORF dazu
                laenge_as = (ende - start) // 3 - 1  # ohne Stopp
                if laenge_as >= min_as:
                    orfs.append((rahmen, start, ende))
                start = None
    return orfs


def finde_orfs(dna, startcodons=STARTCODONS, stoppcodons=STOPPCODONS, min_as=MIN_AMINOSAEUREN):
    """Durchsucht alle 6 Leseraster. Positionen werden biologisch angegeben (ab 1, Ende eingeschlossen)."""
    L = len(dna)
    ergebnisse = []

    for rahmen, s, e in finde_orfs_in_strang(dna, startcodons, stoppcodons, min_as):
        ergebnisse.append({"strang": "+", "rahmen": rahmen + 1,
                           "start": s + 1, "ende": e,
                           "dna": dna[s:e]})

    gegenstrang = reverse_complement(dna)
    for rahmen, s, e in finde_orfs_in_strang(gegenstrang, startcodons, stoppcodons, min_as):
        # Position im Gegenstrang zurückrechnen: Index k im Gegenstrang = Position L - k im Originalstrang (ab 1)
        ergebnisse.append({"strang": "-", "rahmen": rahmen + 1,
                           "start": L - s, "ende": L - e + 1,
                           "dna": gegenstrang[s:e]})

    for orf in ergebnisse:
        orf["protein"] = uebersetze(orf["dna"]).rstrip("*")
    ergebnisse.sort(key=lambda orf: min(orf["start"], orf["ende"]))
    return ergebnisse


def zeige_orfs(orfs):
    print(f"{'Nr':>3}  {'Strang':<6} {'Rahmen':<6} {'Position':<14} {'bp':>5} {'AS':>5}  Protein (Anfang)")
    for nr, orf in enumerate(orfs, start=1):
        if orf["strang"] == "+":
            position = f"{orf['start']}-{orf['ende']}"
        else:
            position = f"c{orf['start']}-{orf['ende']}"     # c = complement, wie bei NCBI
        print(f"{nr:>3}  {orf['strang']:<6} {orf['rahmen']:<6} {position:<14} {len(orf['dna']):>5} "
              f"{len(orf['protein']):>5}  {orf['protein'][:30]}...")


def lade_pPCP1():
    for lokal, online in DATEIEN.items():
        if not os.path.exists(lokal):
            urllib.request.urlretrieve(BASIS_URL + online, lokal)
    plasmid = list(lese_fasta("pPCP1_plasmid.fasta").values())[0]
    gene = lese_fasta("pPCP1_gene.fasta")
    return plasmid, gene


def vergleiche_mit_annotation(orfs, gene):
    """Vergleicht gefundene ORFs mit den NCBI-Genen. Gleiches Ende (Stoppcodon) = gleiches Gen."""
    print("\nVergleich mit den 10 bei NCBI annotierten Genen:")
    print(f"{'NCBI-Position':<14} {'Start':<5} {'Ergebnis':<44} Gen")
    gefunden = 0
    for kopf in gene:
        position = kopf.split(":")[1].split(" ")[0]          # z. B. "87-1109" oder "c5888-4815"
        beschreibung = kopf.split(" ", 1)[1].split(" [")[0]
        strang = "-" if position.startswith("c") else "+"
        a, b = position.lstrip("c").split("-")
        start, ende = int(a), int(b)
        startcodon = gene[kopf][:3]

        treffer = None
        for orf in orfs:
            if orf["strang"] == strang and orf["ende"] == ende:
                treffer = orf
        if treffer is None:
            ergebnis = "❌ nicht gefunden"
        elif treffer["start"] == start:
            ergebnis = "✅ exakt gefunden"
            gefunden = gefunden + 1
        else:
            differenz = abs(treffer["start"] - start)
            richtung = "länger" if len(treffer["dna"]) > len(gene[kopf]) else "kürzer"
            ergebnis = f"🟡 gefunden, Start {differenz} bp anders ({richtung})"
            gefunden = gefunden + 1
        print(f"{position:<14} {startcodon:<5} {ergebnis:<44} {beschreibung}")
    print(f"\n{gefunden} von {len(gene)} Genen wiedergefunden. ORFs insgesamt: {len(orfs)}")


if EIGENE_SEQUENZ.strip():
    dna = "".join(EIGENE_SEQUENZ.split()).upper().replace("U", "T")
    print(f"Eigene Sequenz: {len(dna)} bp\n")
    orfs = finde_orfs(dna)
    zeige_orfs(orfs)
else:
    plasmid, gene = lade_pPCP1()
    print(f"Plasmid pPCP1 (NCBI NC_005816.1): {len(plasmid)} bp")
    print(f"Startcodons: {STARTCODONS}, Mindestlänge: {MIN_AMINOSAEUREN} Aminosäuren\n")
    orfs = finde_orfs(plasmid)
    zeige_orfs(orfs)
    vergleiche_mit_annotation(orfs, gene)

## E5. Stammbaum: UPGMA und Neighbor-Joining
Lädt 12 Orchideen-Sequenzen und rechnet ca. 10–30 Sekunden.

In [ ]:
# 🌳 Stammbaum aus DNA-Sequenzen: Alignment → Distanzen → Baum (UPGMA und Neighbor-Joining)
#
# Ablauf:
#   1. Jedes Sequenzpaar wird mit Needleman-Wunsch aligniert (dein Code aus Teil 1).
#   2. Aus jedem Alignment wird eine evolutionäre Distanz berechnet (p-Distanz, Jukes-Cantor-Korrektur).
#   3. Aus der Distanzmatrix wird ein Baum gebaut: UPGMA und Neighbor-Joining.
#
# Beispieldaten: 5.8S-rRNA + ITS1/ITS2 von 12 Frauenschuh-Orchideen (Unterfamilie Cypripedioideae),
# 4 Gattungen. Quelle: GenBank (Accession-Nummern stehen in der FASTA-Datei), ausgewählt aus dem
# Beispieldatensatz "ls_orchid.fasta" des Biopython-Projekts.
#
# So benutzt du es: Ganzen Code in eine Colab-Zelle kopieren und ausführen (dauert ca. 15–30 s).

import math
import os
import urllib.request

# ============================================================
# EINSTELLUNGEN
# ============================================================
FASTA_DATEI = "orchideen_ITS.fasta"
FASTA_URL = "https://raw.githubusercontent.com/galaxytosst/Bioinfo/main/03_phylogenie/orchideen_ITS.fasta"
AUSSENGRUPPE = ["Cypripedium_calceolus", "Cypripedium_acaule", "Cypripedium_reginae"]   # zum Bewurzeln des NJ-Baums
KORREKTUR = "jukes-cantor"               # "jukes-cantor" oder "p" (unkorrigierte p-Distanz)

MATCH, MISMATCH, GAP = 1, -1, -2
# ============================================================


# ---------- Sequenzen einlesen ----------
def lese_fasta(dateiname):
    sequenzen = {}
    name = None
    with open(dateiname) as f:
        for zeile in f:
            zeile = zeile.strip()
            if zeile == "":
                continue
            if zeile.startswith(">"):
                name = zeile[1:].split()[0]      # nur das erste Wort als Name
                sequenzen[name] = ""
            else:
                sequenzen[name] = sequenzen[name] + zeile.upper()
    return sequenzen


# ---------- 1. Alignment (Needleman-Wunsch aus Teil 1, kompakt) ----------
def needleman_wunsch(s1, s2, match=MATCH, mismatch=MISMATCH, gap=GAP):
    n, m = len(s1), len(s2)
    F = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        F[i][0] = i * gap
    for j in range(1, m + 1):
        F[0][j] = j * gap
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = F[i - 1][j - 1] + (match if s1[i - 1] == s2[j - 1] else mismatch)
            F[i][j] = max(diag, F[i - 1][j] + gap, F[i][j - 1] + gap)
    a1, a2 = "", ""
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and F[i][j] == F[i - 1][j - 1] + (match if s1[i - 1] == s2[j - 1] else mismatch):
            a1, a2, i, j = s1[i - 1] + a1, s2[j - 1] + a2, i - 1, j - 1
        elif i > 0 and F[i][j] == F[i - 1][j] + gap:
            a1, a2, i = s1[i - 1] + a1, "-" + a2, i - 1
        else:
            a1, a2, j = "-" + a1, s2[j - 1] + a2, j - 1
    return a1, a2


# ---------- 2. Distanz ----------
def distanz(a1, a2, korrektur=KORREKTUR):
    """p-Distanz = Anteil verschiedener Basen an allen Positionen ohne Lücke.
    Jukes-Cantor korrigiert für Mehrfachmutationen an derselben Stelle: d = -3/4 · ln(1 - 4/3 · p)."""
    vergleichbar = 0
    verschieden = 0
    for x, y in zip(a1, a2):
        if x == "-" or y == "-":
            continue                      # Lücken werden hier nicht gezählt
        vergleichbar = vergleichbar + 1
        if x != y:
            verschieden = verschieden + 1
    p = verschieden / vergleichbar
    if korrektur == "p":
        return p
    if p >= 0.75:
        return float("inf")              # Sequenzen sind "gesättigt", Distanz nicht schätzbar
    return -0.75 * math.log(1 - 4 / 3 * p)


def distanzmatrix(sequenzen):
    namen = list(sequenzen)
    D = {a: {a: 0.0} for a in namen}
    paare = len(namen) * (len(namen) - 1) // 2
    nr = 0
    for i in range(len(namen)):
        for j in range(i + 1, len(namen)):
            a, b = namen[i], namen[j]
            a1, a2 = needleman_wunsch(sequenzen[a], sequenzen[b])
            D[a][b] = D[b][a] = distanz(a1, a2)
            nr = nr + 1
            if nr % 11 == 0 or nr == paare:
                print(f"Aligniere Paar {nr}/{paare} ...")
    print("fertig.\n")
    return namen, D


def kurz(name):
    # "Cypripedium_calceolus" → "C.calceolus", "Mexipedium_xerophyticum" → "M.xerophyticum"
    teile = name.split("_")
    if len(teile) == 2:
        gattung = teile[0]
        kuerzel = {"Paphiopedilum": "Paph", "Phragmipedium": "Phrag", "Cypripedium": "Cyp", "Mexipedium": "Mex"}
        return kuerzel.get(gattung, gattung[0]) + "." + teile[1]
    return name


def zeige_distanzmatrix(namen, D):
    breite = max(len(kurz(n)) for n in namen)
    print("Distanzmatrix (" + KORREKTUR + "):")
    print(" " * (breite + 2) + "".join(f"{i + 1:>6}" for i in range(len(namen))))
    for i, a in enumerate(namen):
        print(f"{i + 1:>2} {kurz(a):<{breite}}" + "".join(f"{D[a][b]:>6.3f}" for b in namen))
    print()


# ---------- 3a. UPGMA ----------
# Ein Knoten im Baum ist ein Dictionary: {"name": ..., "kinder": [(kindknoten, astlaenge), ...]}
def blatt(name):
    return {"name": name, "kinder": []}


def upgma(namen, D):
    """Fasst immer die zwei ähnlichsten Gruppen zusammen. Annahme: konstante Mutationsrate (molekulare Uhr)."""
    gruppen = {n: {"knoten": blatt(n), "groesse": 1, "hoehe": 0.0} for n in namen}
    D = {a: dict(D[a]) for a in D}                       # Kopie, damit das Original erhalten bleibt
    zaehler = 0
    while len(gruppen) > 1:
        liste = list(gruppen)
        a, b = min(((x, y) for i, x in enumerate(liste) for y in liste[i + 1:]), key=lambda p: D[p[0]][p[1]])
        hoehe = D[a][b] / 2
        A, B = gruppen.pop(a), gruppen.pop(b)
        zaehler = zaehler + 1
        neu = f"U{zaehler}"
        knoten = {"name": None, "kinder": [(A["knoten"], hoehe - A["hoehe"]), (B["knoten"], hoehe - B["hoehe"])]}
        D[neu] = {neu: 0.0}
        for k in gruppen:                                # Durchschnitt, gewichtet nach Gruppengröße
            d = (D[a][k] * A["groesse"] + D[b][k] * B["groesse"]) / (A["groesse"] + B["groesse"])
            D[neu][k] = D[k][neu] = d
        gruppen[neu] = {"knoten": knoten, "groesse": A["groesse"] + B["groesse"], "hoehe": hoehe}
    return list(gruppen.values())[0]["knoten"]


# ---------- 3b. Neighbor-Joining ----------
def neighbor_joining(namen, D, aussengruppe):
    """Braucht keine molekulare Uhr. Ergibt einen unbewurzelten Baum, der hier an der Außengruppe bewurzelt wird."""
    D = {a: dict(D[a]) for a in D}
    aktiv = list(namen)
    nachbarn = {n: [] for n in namen}                    # ungerichteter Graph: Knoten → [(Nachbar, Länge)]
    zaehler = 0
    while len(aktiv) > 2:
        n = len(aktiv)
        r = {i: sum(D[i][k] for k in aktiv) for i in aktiv}
        i, j = min(((x, y) for a, x in enumerate(aktiv) for y in aktiv[a + 1:]),
                   key=lambda p: (n - 2) * D[p[0]][p[1]] - r[p[0]] - r[p[1]])
        li = D[i][j] / 2 + (r[i] - r[j]) / (2 * (n - 2))
        lj = D[i][j] - li
        zaehler = zaehler + 1
        u = f"N{zaehler}"
        nachbarn[u] = [(i, li), (j, lj)]
        nachbarn[i].append((u, li))
        nachbarn[j].append((u, lj))
        D[u] = {u: 0.0}
        for k in aktiv:
            if k not in (i, j):
                D[u][k] = D[k][u] = (D[i][k] + D[j][k] - D[i][j]) / 2
        aktiv = [k for k in aktiv if k not in (i, j)] + [u]
    a, b = aktiv
    nachbarn[a].append((b, D[a][b]))
    nachbarn[b].append((a, D[a][b]))

    def teilbaum(knoten, herkunft):
        kinder = [(teilbaum(nb, knoten), l) for nb, l in nachbarn[knoten] if nb != herkunft]
        return {"name": knoten if knoten in namen else None, "kinder": kinder}

    def blaetter_hinter(knoten, herkunft):
        # alle Blätter, die man von 'knoten' aus erreicht, ohne zurück zu 'herkunft' zu gehen
        if knoten in namen:
            return {knoten}
        menge = set()
        for nb, _ in nachbarn[knoten]:
            if nb != herkunft:
                menge = menge | blaetter_hinter(nb, knoten)
        return menge

    # Wurzel in die Mitte des Asts setzen, der die Außengruppe vom Rest trennt
    ziel = set(aussengruppe)
    for u in nachbarn:
        for v, laenge in nachbarn[u]:
            if blaetter_hinter(v, u) == ziel:
                return {"name": None, "kinder": [(teilbaum(v, u), laenge / 2), (teilbaum(u, v), laenge / 2)]}
    print("⚠️ Die Außengruppe bildet im Baum keine eigene Gruppe. Bewurzelt wird an ihrem ersten Mitglied.")
    erstes = aussengruppe[0]
    elternteil, laenge = nachbarn[erstes][0]
    return {"name": None, "kinder": [(blatt(erstes), laenge / 2), (teilbaum(elternteil, erstes), laenge / 2)]}


# ---------- Ausgabe ----------
def newick(knoten, laenge=None):
    if knoten["kinder"]:
        text = "(" + ",".join(newick(k, l) for k, l in knoten["kinder"]) + ")"
    else:
        text = knoten["name"]
    if laenge is not None:
        text = text + f":{laenge:.4f}"
    return text


def zeige_baum_text(knoten, praefix="", laenge=None, letzter=True, wurzel=True):
    beschriftung = kurz(knoten["name"]) if knoten["name"] else "┐"
    if wurzel:
        print(beschriftung)
    else:
        print(praefix + ("└── " if letzter else "├── ") + beschriftung + f"  ({laenge:.3f})")
    neues_praefix = praefix + ("" if wurzel else ("    " if letzter else "│   "))
    for nr, (kind, l) in enumerate(knoten["kinder"]):
        zeige_baum_text(kind, neues_praefix, l, nr == len(knoten["kinder"]) - 1, False)


def blaetter(knoten):
    if not knoten["kinder"]:
        return [knoten["name"]]
    ergebnis = []
    for kind, _ in knoten["kinder"]:
        ergebnis = ergebnis + blaetter(kind)
    return ergebnis


def zeichne_baum(knoten, titel, ax):
    """Zeichnet einen rechtwinkligen Baum mit matplotlib."""
    farben = {"Cypripedium": "tab:green", "Paphiopedilum": "tab:purple", "Phragmipedium": "tab:orange", "Mexipedium": "tab:red"}
    y_position = {name: i for i, name in enumerate(blaetter(knoten))}

    def zeichne(k, x):
        if not k["kinder"]:
            y = y_position[k["name"]]
            farbe = farben.get(k["name"].split("_")[0], "black")
            ax.text(x + 0.002, y, " " + k["name"].replace("_", " "), va="center", fontsize=9, color=farbe, style="italic")
            return y
        ys = []
        for kind, l in k["kinder"]:
            y_kind = zeichne(kind, x + l)
            ax.plot([x, x + l], [y_kind, y_kind], color="black", lw=1.2)
            ys.append(y_kind)
        ax.plot([x, x], [min(ys), max(ys)], color="black", lw=1.2)
        return sum(ys) / len(ys)

    zeichne(knoten, 0.0)
    ax.set_title(titel)
    ax.set_yticks([])
    ax.set_xlabel("Distanz (Substitutionen pro Position)")
    for seite in ["left", "right", "top"]:
        ax.spines[seite].set_visible(False)
    x_max = ax.get_xlim()[1]
    ax.set_xlim(0, x_max * 1.6)                         # Platz für die Namen


# ---------- Hauptprogramm ----------
if not os.path.exists(FASTA_DATEI):
    urllib.request.urlretrieve(FASTA_URL, FASTA_DATEI)
sequenzen = lese_fasta(FASTA_DATEI)
print(f"{len(sequenzen)} Sequenzen geladen:")
for name, seq in sequenzen.items():
    print(f"  {name:<32} {len(seq)} bp")
print()

namen, D = distanzmatrix(sequenzen)
zeige_distanzmatrix(namen, D)

baum_upgma = upgma(namen, D)
baum_nj = neighbor_joining(namen, D, AUSSENGRUPPE)

print("=== UPGMA ===")
zeige_baum_text(baum_upgma)
print("\nNewick:", newick(baum_upgma) + ";")

print("\n=== Neighbor-Joining (bewurzelt mit Außengruppe: " + ", ".join(kurz(n) for n in AUSSENGRUPPE) + ") ===")
zeige_baum_text(baum_nj)
print("\nNewick:", newick(baum_nj) + ";")

try:
    import matplotlib.pyplot as plt
    fig, achsen = plt.subplots(1, 2, figsize=(15, 6))
    zeichne_baum(baum_upgma, "UPGMA", achsen[0])
    zeichne_baum(baum_nj, "Neighbor-Joining", achsen[1])
    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(matplotlib nicht installiert, daher keine Grafik.)")